In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='f63990a5-d117-40f7-9fa3-4db09eac96f3'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "langchain-mcp-adapters library"}'}, '__gemini_function_call_thought_signatures__': {'d3cdc1aa-db6c-42dd-ad12-9693cd38badc': 'Cq8BAQw51scmDUQQTqoqhTet0dSo2VNDMyNV1sZi9OgiWZh8+Pq0FBKgokmlMg1hMUb3OySk5M52YRwSjiMZiaNebQxZxe6NKXtn5jXlZIco7MKg2VHKCKl3c8XuTkcM2bSvA/rotrYW5v7+AjFGxuPNpNYjU653lphgknXYQ4pDTyCX55qBA0TyE/lfv8ZkvWGs6jttHjUb75V/Ug9/Rfr64DZy/lClE5QIZiJEV7SmcA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e54b2-fdae-7012-bfc9-e7d642d08078-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapters library'}, 'id': 'd3cdc1aa-db6c-42dd-ad12-9693cd38badc', 'type': 'tool

## Online MCP

In [13]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()
pprint(tools)

[StructuredTool(name='get_current_time', description='Get current time in a specific timezones', args_schema={'type': 'object', 'properties': {'timezone': {'type': 'string', 'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no timezone provided by the user."}}, 'required': ['timezone']}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x00000194FE40B880>),
 StructuredTool(name='convert_time', description='Convert time between timezones', args_schema={'type': 'object', 'properties': {'source_timezone': {'type': 'string', 'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'America/New_York' as local timezone if no source timezone provided by the user."}, 'time': {'type': 'string', 'description': 'Time to convert in 24-hour format (HH:MM)'}, 'target_timezone': {'type': 'string', 'description': "Target IANA ti

In [10]:
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=tools,
)

In [15]:
question = HumanMessage(content="What time is it in india?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it in india?', additional_kwargs={}, response_metadata={}, id='18039abd-87cc-4e53-b2ce-65b0302422ac'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Kolkata"}'}, '__gemini_function_call_thought_signatures__': {'2e67bcd8-d033-4d54-8654-4316af1bbc71': 'CuoBAQw51scVhVIvCtbOQ53P/Z6H8uH3nw5Z3zgb/BGmwmJcfQhmCn4T22dEeow7VuQtfrDRVvZCFhRHKxOUcUqyV9fopc7EZ9m04TAITtI4BXbpZp6/s9Hp9TJ1IxDgLfLH3w1ekKUBtgo0I0QSU9cMUjTK5xmdBt5OBygGFxXb/RU9vcjebkcmg1YDRv7dAJuIeXYb0+Uwd/1lg9LZFX5zccHRHnAQQ5Y3SDt2XSbfsbB6cJ8pHOOw0w3zsXXaXA6G93FQUp6EOCJQ8iuNzuc3qZcaELr+KpX2pErR9tRzMiI4WhzV7eKzccw8'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e54b8-8f5b-7880-943b-f14cb293d4ba-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Kolkata'}, 'id': '2e67bcd8-d033-4d5